# QLoRA SFT of the Builder's diagnosis model

Fine-tunes **Qwen/Qwen2.5-Coder-7B-Instruct** (the weights behind Ollama's `qwen2.5-coder:7b`) on the
Builder's fix-diagnosis examples (`experiments/finetune/sft.jsonl`): 4-bit base weights, LoRA
adapters, loss on the assistant answers only. Exports a **GGUF Q4_K_M** and an Ollama **Modelfile**
for `qwen-builder-sft`.

- Runtime: Google Colab, **T4 GPU** (Runtime > Change runtime type > T4 GPU).
- Data: only the 9 `split=memory` faults (the repo's tests check that no held-out test fault is in it).
- Afterwards, on your machine: `ollama create qwen-builder-sft -f Modelfile`, then run the fix loop
  with `--model qwen-builder-sft` (last cell).

In [ ]:
%%capture
# Unsloth: 4-bit loading, LoRA, chat templates, and GGUF export (pulls compatible torch/transformers/trl/peft)
!pip install unsloth

In [ ]:
import importlib.metadata as md_
for pkg in ("unsloth", "torch", "transformers", "trl", "peft", "bitsandbytes"):
    print(f"{pkg:13} {md_.version(pkg)}")   # record these with the results

## Hyperparameters

All settings in one place; the rest of the notebook reads them.

In [ ]:
# --- data ---
DATA_REF = "main"              # git branch or commit of fedimdaini/builder-agent to read sft.jsonl from
DATA_URL = f"https://raw.githubusercontent.com/fedimdaini/builder-agent/{DATA_REF}/experiments/finetune/sft.jsonl"  # the public repo: no token needed
MAX_PER_CASE = 4               # at most 4 examples per fault (human, sampled, rationalized, written first): no fault dominates
MAX_SEQ_LEN = 6144             # tokens per example; the longest prompt is about 5k tokens, longer ones are dropped and counted

# --- model ---
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"   # base model: the same weights as Ollama's qwen2.5-coder:7b
LOAD_IN_4BIT = True            # QLoRA: base weights in 4-bit NF4, so the 7B model fits in a 16 GB T4

# --- LoRA ---
LORA_R = 16                    # rank of the adapter matrices: how much the adapter can learn
LORA_ALPHA = 16                # scale of the adapter's update (alpha / r = 1)
LORA_DROPOUT = 0.0             # no dropout: Unsloth's fast path, and the run is only a few dozen steps
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]  # every attention and MLP projection gets an adapter

# --- training ---
NUM_EPOCHS = 3                 # passes over the data (about 30 examples, so about 23 updates)
BATCH_SIZE = 1                 # examples per GPU step: the prompts are long for a T4
GRAD_ACCUM = 4                 # GPU steps per update: effective batch of 4
LEARNING_RATE = 2e-4           # usual LoRA learning rate (only the adapters are trained)
WARMUP_STEPS = 3               # the learning rate ramps up over the first updates
LR_SCHEDULER = "linear"        # then decays linearly to 0
WEIGHT_DECAY = 0.01            # light regularization of the adapter weights
SEED = 42                      # the same seed as the Builder's Ollama runs

# --- export ---
GGUF_QUANT = "q4_k_m"          # quantization of the exported model: the same level as Ollama's qwen2.5-coder:7b
OLLAMA_NAME = "qwen-builder-sft"   # the name to use with `ollama create` and the fix loop's --model
OUT_DIR = "qwen-builder-sft"   # local folder for the adapters and the GGUF

## Data: download, cap per fault, check

In [ ]:
import hashlib, json, urllib.request
from collections import Counter

raw = urllib.request.urlopen(DATA_URL).read()
print("sft.jsonl sha256", hashlib.sha256(raw).hexdigest(), f"({DATA_REF})")   # record it: identifies the data
rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]

# the same rule as builder_agent.finetune.cap_per_case
CAP_ORDER = ("human", "sampled", "rationalized", "written")
capped = []
for case_id in dict.fromkeys(r["case_id"] for r in rows):
    case_rows = [r for r in rows if r["case_id"] == case_id]
    capped += sorted(case_rows, key=lambda r: CAP_ORDER.index(r["source"]))[:MAX_PER_CASE]

assert all(r["messages"][-1]["role"] == "assistant" for r in capped)
assert not any("HINT (for this answer only)" in json.dumps(r) for r in capped)   # never the rationalization hint
print(f"{len(rows)} examples -> {len(capped)} after the cap of {MAX_PER_CASE} per fault")
print("per fault :", dict(Counter(r["case_id"] for r in capped)))
print("per source:", dict(Counter(r["source"] for r in capped)))
print("per family:", dict(Counter(r["family"] for r in capped)))

## Model: 4-bit base + LoRA adapters

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, load_in_4bit=LOAD_IN_4BIT, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, target_modules=TARGET_MODULES,
    bias="none", use_gradient_checkpointing="unsloth", random_state=SEED)
model.print_trainable_parameters()

## Format: Qwen chat template, length check

In [ ]:
from datasets import Dataset

def to_text(r):
    return tokenizer.apply_chat_template(r["messages"], tokenize=False)   # ChatML: <|im_start|>role ... <|im_end|>

texts = [to_text(r) for r in capped]
lengths = [len(tokenizer(t)["input_ids"]) for t in texts]
keep = [t for t, n in zip(texts, lengths) if n <= MAX_SEQ_LEN]
print(f"tokens per example: min {min(lengths)}, max {max(lengths)}; dropped as too long: {len(texts) - len(keep)}")
dataset = Dataset.from_dict({"text": keep})

## Train on the assistant answers only

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN, packing=False,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS,
        lr_scheduler_type=LR_SCHEDULER, weight_decay=WEIGHT_DECAY, optim="adamw_8bit",
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),   # a T4 has no bf16
        logging_steps=1, seed=SEED, output_dir=OUT_DIR, report_to="none"))

# loss only on the answer: everything up to "<|im_start|>assistant\n" is masked (label -100)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

# check the mask on the first example: only the JSON answer should remain
labels = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
print("trained on:", tokenizer.decode(labels)[:300])

In [ ]:
stats = trainer.train()
print(stats.metrics)

## Loss

In [ ]:
import csv
import matplotlib.pyplot as plt

log = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
with open(f"{OUT_DIR}/loss.csv", "w", newline="") as f:
    csv.writer(f).writerows([("step", "loss"), *log])
plt.plot([s for s, _ in log], [l for _, l in log], marker="o")
plt.xlabel("update"); plt.ylabel("training loss (answers only)"); plt.title(OLLAMA_NAME); plt.grid(alpha=0.3)
plt.savefig(f"{OUT_DIR}/loss.png", dpi=120, bbox_inches="tight"); plt.show()

## Quick look at one answer

A training example, so this only checks that the model answers in the expected JSON shape. The real
evaluation is the fix loop on the held-out faults, on your machine.

In [ ]:
FastLanguageModel.for_inference(model)
prompt = tokenizer.apply_chat_template(capped[0]["messages"][:-1], tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Export: GGUF Q4_K_M and an Ollama Modelfile

In [ ]:
import glob, os, shutil

model.save_pretrained(f"{OUT_DIR}/lora")             # the adapters alone (small), to keep
tokenizer.save_pretrained(f"{OUT_DIR}/lora")
model.save_pretrained_gguf(OUT_DIR, tokenizer, quantization_method=GGUF_QUANT)   # merges, converts, quantizes

gguf_name = f"{OLLAMA_NAME}.{GGUF_QUANT.upper()}.gguf"

# the Modelfile first: it only names the GGUF, so a failed search below no longer skips it
# Qwen's ChatML template; the Builder sends system + user messages and asks for JSON (Ollama "format")
modelfile = f"""FROM ./{gguf_name}
TEMPLATE \"\"\"{{{{- range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
\"\"\"
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
PARAMETER num_ctx 8192
"""
with open("Modelfile", "w") as f:
    f.write(modelfile)
print(modelfile)

# Unsloth writes the GGUF to {OUT_DIR}_gguf/Qwen2.5-Coder-7B-Instruct.Q4_K_M.gguf (not into OUT_DIR).
# Find it there and move it (a 4.7 GB copy can fill Colab's disk); a rerun finds the moved file.
if not os.path.exists(gguf_name):
    ggufs = [g for d in (f"{OUT_DIR}_gguf", OUT_DIR) for g in glob.glob(f"{d}/**/*.gguf", recursive=True)]
    gguf = [g for g in ggufs if GGUF_QUANT.upper() in os.path.basename(g).upper()]
    assert gguf, f"no {GGUF_QUANT.upper()} .gguf in {OUT_DIR}_gguf/ or {OUT_DIR}/: {ggufs}"
    shutil.move(gguf[0], gguf_name)
print(gguf_name, f"{os.path.getsize(gguf_name) / 1e9:.1f} GB")

In [ ]:
# download: the GGUF (about 4.7 GB), the Modelfile, the loss curve and the adapters
import zipfile
from google.colab import files

with zipfile.ZipFile(f"{OLLAMA_NAME}-extras.zip", "w") as z:
    for p in ["Modelfile", f"{OUT_DIR}/loss.csv", f"{OUT_DIR}/loss.png", *glob.glob(f"{OUT_DIR}/lora/*")]:
        z.write(p)
files.download(f"{OLLAMA_NAME}-extras.zip")
files.download(gguf_name)   # large: if the browser download stalls, copy it to Google Drive instead

## On your machine

```bash
# put the .gguf next to the Modelfile, then:
ollama create qwen-builder-sft -f Modelfile
ollama list   # shows qwen-builder-sft:latest

# the fix loop on a held-out fault, with the fine-tuned model
python -m builder_agent.fix ../taxi-trip-regression --slots tests/gold/taxi_slots.json \
  --expected tests/gold/taxi_expected.json --prompt diagnose_v3 --retriever advanced \
  --model qwen-builder-sft --case tests/faults/generated/python_3_13/case.json \
  --out experiments/fixes/gen-011-python_3_13__diagnose_v3-advanced__qwen-builder-sft
```